# 1. Importação de Dados do Kaggle para o BigQuery
Dataset: Mercado de Games

Este notebook utiliza a biblioteca `kagglehub` para baixar os arquivos originais (.csv) do Kaggle em tempo real e em seguida enviá-los diretamente para o Google BigQuery.

## 1. Importação das Bibliotecas

In [ ]:
import pandas as pd
import os
import kagglehub
from google.cloud import bigquery

# Se você estiver usando o Google Colab, descomente as linhas abaixo para autenticar sua conta GCP
# from google.colab import auth
# auth.authenticate_user()

## 2. Autenticação no BigQuery

In [ ]:
# Substitua pelas suas credenciais reais do Google Cloud Platform
project_id = 'seu_projeto'
dataset_id = 'seu_dataset'

client = bigquery.Client(project=project_id)

## 3. Baixar os Dados Diretamente do Kaggle

In [ ]:
print("Iniciando o download da base de dados mais recente do Kaggle...")

# Faz o download automático e retorna o caminho onde os arquivos foram salvos na máquina/ambiente
data_path = kagglehub.dataset_download("gabrigoncam/video-game-dataset-multidimensional")

print("Arquivos baixados na pasta local:", data_path)

# Extrair o caminho de todos os CSVs fornecidos no dataset
all_files = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

print(f"\nEncontrados {len(all_files)} arquivos .csv prontos para envio ao BigQuery.")

## 4. Subir Tudo para o BigQuery

In [ ]:
for file_path in all_files:
    # Extrair o nome do arquivo para usar como nome da tabela no BigQuery
    file_name = os.path.basename(file_path)
    table_name = file_name.replace('.csv', '')
    
    table_id = f"{project_id}.{dataset_id}.{table_name}"
    
    print(f"Enviando [{file_name}] para a tabela [{table_id}]...")
    
    df = pd.read_csv(file_path)
    
    # Substituir (truncate) a tabela caso ela já exista no dataset
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    
    # Dispara o job pro BQ
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    
    print(f"\t>> ✅ Sucesso! Tabela {table_name} carregada.\n")

print("Todos os dados foram enviados para o Google BigQuery com sucesso!")